# random_forest_v1

Compact z-score random-forest benchmark with configurable train / validation / test time splits.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG = {
    "dataset_path": PROJECT_ROOT / "data" / "datasets" / "stock_panel_nine_tickers_session_aligned_raw.csv",
    "excluded_tickers": ["NFLX"],
    "neutral_band": 0.005,
    "test_size": 0.20,
    "validation_fraction_within_pretest": 0.15,
    "min_validation_dates": 20,
    "gap_days": 1,
    "rf_params": {
        "n_estimators": 500,
        "max_depth": 10,
        "min_samples_leaf": 10,
        "min_samples_split": 25,
        "max_features": 0.5,
        "class_weight": "balanced_subsample",
        "random_state": 42,
        "n_jobs": -1,
    },
}

FEATURE_SETS = {
    "Model A - price only": [
        "return_1d",
        "return_5d",
        "return_20d",
        "rolling_volatility_20d",
    ],
    "Model B - price + volume": [
        "return_1d",
        "return_5d",
        "return_20d",
        "rolling_volatility_20d",
        "volume_zscore_20d",
    ],
    "Model C - price + volume + GDELT": [
        "return_1d",
        "return_5d",
        "return_20d",
        "rolling_volatility_20d",
        "volume_zscore_20d",
        "gdelt_sentiment_zscore_30d",
        "gdelt_article_count_zscore_30d",
    ],
    "Model D - price + volume + GDELT + Reddit": [
        "return_1d",
        "return_5d",
        "return_20d",
        "rolling_volatility_20d",
        "volume_zscore_20d",
        "gdelt_sentiment_zscore_30d",
        "gdelt_article_count_zscore_30d",
        "reddit_sentiment_zscore_30d",
        "reddit_comment_count_zscore_30d",
    ],
    "Model E - price + volume + all alternative data": [
        "return_1d",
        "return_5d",
        "return_20d",
        "rolling_volatility_20d",
        "volume_zscore_20d",
        "gdelt_sentiment_zscore_30d",
        "gdelt_article_count_zscore_30d",
        "reddit_sentiment_zscore_30d",
        "reddit_comment_count_zscore_30d",
        "google_trends_zscore_90d",
    ],
}

In [ ]:
def add_trailing_zscore(frame: pd.DataFrame, source_col: str, output_col: str, window: int) -> None:
    grouped = frame.groupby("ticker")[source_col]
    rolling_mean = grouped.transform(lambda s: s.shift(1).rolling(window).mean())
    rolling_std = grouped.transform(lambda s: s.shift(1).rolling(window).std())
    frame[output_col] = (frame[source_col] - rolling_mean) / rolling_std.replace(0.0, np.nan)


def build_feature_frame(raw_df: pd.DataFrame, neutral_band: float) -> pd.DataFrame:
    frame = raw_df.copy().sort_values(["ticker", "date"]).reset_index(drop=True)
    frame["comm_reddit_vader_mean"] = frame["comm_reddit_vader_mean"].fillna(0.0)
    frame["comm_reddit_posts"] = frame["comm_reddit_posts"].fillna(0.0)

    price_group = frame.groupby("ticker")["stock_price"]
    frame["return_1d"] = price_group.pct_change(1)
    frame["return_5d"] = price_group.pct_change(5)
    frame["return_20d"] = price_group.pct_change(20)
    frame["rolling_volatility_20d"] = (
        frame.groupby("ticker")["return_1d"].transform(lambda s: s.shift(1).rolling(20).std())
    )

    add_trailing_zscore(frame, "stock_volume", "volume_zscore_20d", 20)
    add_trailing_zscore(frame, "gdelt_sentiment_score", "gdelt_sentiment_zscore_30d", 30)
    add_trailing_zscore(frame, "gdelt_articles", "gdelt_article_count_zscore_30d", 30)
    add_trailing_zscore(frame, "comm_reddit_vader_mean", "reddit_sentiment_zscore_30d", 30)
    add_trailing_zscore(frame, "comm_reddit_posts", "reddit_comment_count_zscore_30d", 30)
    add_trailing_zscore(frame, "google_trends_score", "google_trends_zscore_90d", 90)

    frame["future_return_1d"] = price_group.shift(-1) / frame["stock_price"] - 1.0
    frame["target"] = np.select(
        [
            frame["future_return_1d"] < -neutral_band,
            frame["future_return_1d"] > neutral_band,
        ],
        [0, 1],
        default=np.nan,
    )
    frame["target_available"] = frame["future_return_1d"].notna()
    frame["is_neutral"] = frame["target_available"] & frame["future_return_1d"].abs().le(neutral_band)
    return frame


def make_train_validation_test_split_by_date(
    frame: pd.DataFrame,
    test_size: float,
    validation_fraction_within_pretest: float,
    min_validation_dates: int,
    gap_days: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    unique_dates = sorted(frame["date"].drop_duplicates())

    test_start_idx = int(np.floor(len(unique_dates) * (1.0 - test_size)))
    test_start_idx = min(max(test_start_idx, 2), len(unique_dates) - 1)
    pretest_end_idx = max(test_start_idx - gap_days, 1)
    pretest_dates = unique_dates[:pretest_end_idx]

    validation_size = int(np.floor(len(pretest_dates) * validation_fraction_within_pretest))
    validation_size = max(min_validation_dates, validation_size)
    validation_size = min(max(validation_size, 1), len(pretest_dates) - 1)

    validation_start_idx = len(pretest_dates) - validation_size
    train_end_idx = max(validation_start_idx - gap_days, 1)

    train_dates = unique_dates[:train_end_idx]
    validation_dates = pretest_dates[validation_start_idx:]
    test_dates = unique_dates[test_start_idx:]

    train_df = frame[frame["date"].isin(train_dates)].copy()
    validation_df = frame[frame["date"].isin(validation_dates)].copy()
    test_df = frame[frame["date"].isin(test_dates)].copy()
    return train_df, validation_df, test_df


def safe_auc(y_true: pd.Series, p_up: np.ndarray) -> float:
    if y_true.nunique() < 2:
        return np.nan
    return float(roc_auc_score(y_true, p_up))


def build_rf_pipeline() -> Pipeline:
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(**CONFIG["rf_params"])),
        ]
    )

In [ ]:
raw_df = pd.read_csv(CONFIG["dataset_path"], parse_dates=["date"])
raw_df = raw_df[~raw_df["ticker"].isin(CONFIG["excluded_tickers"])].copy()
raw_df = raw_df.sort_values(["ticker", "date"]).reset_index(drop=True)

feature_df = build_feature_frame(raw_df, neutral_band=CONFIG["neutral_band"])
modeled_df = feature_df[feature_df["target"].isin([0.0, 1.0])].copy()
modeled_df["target"] = modeled_df["target"].astype(int)

train_df, validation_df, test_df = make_train_validation_test_split_by_date(
    modeled_df,
    test_size=CONFIG["test_size"],
    validation_fraction_within_pretest=CONFIG["validation_fraction_within_pretest"],
    min_validation_dates=CONFIG["min_validation_dates"],
    gap_days=CONFIG["gap_days"],
)

split_summary_df = pd.DataFrame(
    [
        {"split": "train", "n_rows": len(train_df), "n_dates": train_df["date"].nunique(), "date_min": train_df["date"].min(), "date_max": train_df["date"].max()},
        {"split": "validation", "n_rows": len(validation_df), "n_dates": validation_df["date"].nunique(), "date_min": validation_df["date"].min(), "date_max": validation_df["date"].max()},
        {"split": "test", "n_rows": len(test_df), "n_dates": test_df["date"].nunique(), "date_min": test_df["date"].min(), "date_max": test_df["date"].max()},
    ]
)

split_summary_df

In [ ]:
results = []
feature_importance_frames = []

for model_name, features in FEATURE_SETS.items():
    X_train = train_df[features]
    y_train = train_df["target"]
    X_validation = validation_df[features]
    y_validation = validation_df["target"]
    X_train_validation = pd.concat([train_df[features], validation_df[features]], axis=0)
    y_train_validation = pd.concat([train_df["target"], validation_df["target"]], axis=0)
    X_test = test_df[features]
    y_test = test_df["target"]

    validation_pipeline = build_rf_pipeline()
    validation_pipeline.fit(X_train, y_train)
    validation_preds = validation_pipeline.predict(X_validation)
    validation_probs = validation_pipeline.predict_proba(X_validation)[:, 1]

    test_pipeline = build_rf_pipeline()
    test_pipeline.fit(X_train_validation, y_train_validation)
    test_preds = test_pipeline.predict(X_test)
    test_probs = test_pipeline.predict_proba(X_test)[:, 1]

    results.append(
        {
            "model": model_name,
            "n_features": len(features),
            "train_rows": len(X_train),
            "validation_rows": len(X_validation),
            "train_plus_validation_rows": len(X_train_validation),
            "test_rows": len(X_test),
            "validation_accuracy": float(accuracy_score(y_validation, validation_preds)),
            "validation_balanced_accuracy": float(balanced_accuracy_score(y_validation, validation_preds)),
            "validation_auc": safe_auc(y_validation, validation_probs),
            "validation_positive_rate": float(np.mean(validation_preds)),
            "test_accuracy": float(accuracy_score(y_test, test_preds)),
            "test_balanced_accuracy": float(balanced_accuracy_score(y_test, test_preds)),
            "test_auc": safe_auc(y_test, test_probs),
            "test_positive_rate": float(np.mean(test_preds)),
        }
    )

    feature_importance_frames.append(
        pd.DataFrame(
            {
                "model": model_name,
                "feature": features,
                "importance": test_pipeline.named_steps["model"].feature_importances_,
            }
        )
    )

results_df = pd.DataFrame(results).sort_values(
    ["validation_balanced_accuracy", "validation_accuracy", "validation_auc", "model"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

feature_importances_df = (
    pd.concat(feature_importance_frames, ignore_index=True)
    .sort_values(["model", "importance"], ascending=[True, False])
    .reset_index(drop=True)
)

results_df

In [ ]:
feature_importances_df